In [1]:
import pandas as pd
import os
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import quote_plus
from rdflib.namespace import RDF, DC, Namespace
import xml.etree.ElementTree as ET
from lxml import etree
import shutil
import zipfile
import ftplib
import io
import csv
import field_extractor as fe

In [2]:
# Constants
UNZIP_DIR = "selected_data"
FTP_HOST = "download.europeana.eu"
FTP_PATH = "dataset/XML/"
OUTPUT_DIR = "collected_data"

In [3]:
data_ids = []
# extract the dataset_ids
with open('modified_ids.txt', 'r') as f:
    data_ids = f.readlines()
    data_ids = [x.strip() for x in data_ids]

data_ids = ['2021672.zip', '728.zip', '73.zip']
print(data_ids)

['2021672.zip', '728.zip', '73.zip']


In [4]:
# Check if UNZIP_DIR exists, if it does, delete it
if os.path.exists(UNZIP_DIR):
    shutil.rmtree(UNZIP_DIR)

# Check if OUTPUT_DIR exists, if it does, delete it
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

In [5]:
# Create the directories again
os.makedirs(UNZIP_DIR)
os.makedirs(OUTPUT_DIR)

In [6]:
def download_file(ftp_host, ftp_path, filename):
    zip_data = io.BytesIO()

    with ftplib.FTP(ftp_host) as ftp:
        ftp.login()  # Login as anonymous
        ftp.cwd(ftp_path)

        ftp.retrbinary(f'RETR {filename}', zip_data.write)
    
    zip_data.seek(0)
    return zip_data

# Function to unzip a file
def unzip_file(zip_data):
    extracted_files = []  # List to store file content

    with zipfile.ZipFile(zip_data, 'r') as zip_ref:
        for file_info in zip_ref.infolist():
            with zip_ref.open(file_info) as file:
                file_content = file.read()
                extracted_files.append(file_content)

    return extracted_files

def find_tier_information(tree, namespaces):
    # Initialize variables to store tier information
    content_tier = None

    # Find all hasBody elements
    has_body_elements = tree.findall('.//oa:hasBody', namespaces)

    for element in has_body_elements:
        resource = element.get('{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource', '')
        
        # Check for content tier
        if 'contentTier' in resource:
            content_tier = resource.split('contentTier')[-1]

    return content_tier

# Function to download and process a ZIP file
def download_zip(filename):

    try:
        print(f"Starting download and processing for {filename}...")
        
        # Download the ZIP file into memory
        zip_data = download_file(FTP_HOST, FTP_PATH, filename)

        # Unzip and process the file
        extracted_files = unzip_file(zip_data)
        print(f"deleting zip file {filename}")
        del zip_data 

        ## TODO: Clicked docs 

        ## TODO: Sampled docs

        # print the filenames from the extracted files
        print(f"Extracted files: {(extracted_files[0])}")

        # make a subdirectory for the dataset
        dataset_dir = os.path.join(UNZIP_DIR, filename)
        dataset_dir = dataset_dir[:-4]
        if not os.path.exists(dataset_dir):
            os.makedirs(dataset_dir)
        
        # save the extracted files in the subdirectory
        for i, file_content in enumerate(extracted_files):
            with open(os.path.join(dataset_dir, f"{i}.xml"), 'wb') as file:
                file.write(file_content)

        print(f"Finished processing {filename}")
        del extracted_files

    except Exception as e:
        print(f"Error processing {filename}: {e}")

In [7]:
# Use ThreadPoolExecutor to download and process files in parallel
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(download_zip, filename) for filename in data_ids]

    # Use tqdm to show progress as futures are completed
    for future in tqdm(as_completed(futures), total=len(futures), desc="Processing ZIP files"):
        # This will raise exceptions if any occurred during processing
        future.result()

Starting download and processing for 2021672.zip...
Starting download and processing for 728.zip...
Starting download and processing for 73.zip...


Processing ZIP files:  33%|███▎      | 1/3 [00:00<00:01,  1.37it/s]

deleting zip file 2021672.zip
Extracted files: b'<?xml version="1.0" encoding="UTF-8" standalone="yes"?><rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#" xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:dcterms="http://purl.org/dc/terms/" xmlns:edm="http://www.europeana.eu/schemas/edm/" xmlns:owl="http://www.w3.org/2002/07/owl#" xmlns:wgs84_pos="http://www.w3.org/2003/01/geo/wgs84_pos#" xmlns:skos="http://www.w3.org/2004/02/skos/core#" xmlns:rdaGr2="http://rdvocab.info/ElementsGr2/" xmlns:foaf="http://xmlns.com/foaf/0.1/" xmlns:ebucore="http://www.ebu.ch/metadata/ontologies/ebucore/ebucore#" xmlns:doap="http://usefulinc.com/ns/doap#" xmlns:odrl="http://www.w3.org/ns/odrl/2/" xmlns:cc="http://creativecommons.org/ns#" xmlns:ore="http://www.openarchives.org/ore/terms/" xmlns:svcs="http://rdfs.org/sioc/services#" xmlns:oa="http://www.w3.org/ns/oa#" xmlns:dqv="http://www.w3.org/ns/dqv#"><edm:ProvidedCHO rdf:about="http://data.europeana.eu/item/2021672/resource_document_mauri

Processing ZIP files: 100%|██████████| 3/3 [00:01<00:00,  2.40it/s]

deleting zip file 73.zip
Extracted files: b'<?xml version="1.0" encoding="UTF-8" standalone="yes"?><rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#" xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:dcterms="http://purl.org/dc/terms/" xmlns:edm="http://www.europeana.eu/schemas/edm/" xmlns:owl="http://www.w3.org/2002/07/owl#" xmlns:wgs84_pos="http://www.w3.org/2003/01/geo/wgs84_pos#" xmlns:skos="http://www.w3.org/2004/02/skos/core#" xmlns:rdaGr2="http://rdvocab.info/ElementsGr2/" xmlns:foaf="http://xmlns.com/foaf/0.1/" xmlns:ebucore="http://www.ebu.ch/metadata/ontologies/ebucore/ebucore#" xmlns:doap="http://usefulinc.com/ns/doap#" xmlns:odrl="http://www.w3.org/ns/odrl/2/" xmlns:cc="http://creativecommons.org/ns#" xmlns:ore="http://www.openarchives.org/ore/terms/" xmlns:svcs="http://rdfs.org/sioc/services#" xmlns:oa="http://www.w3.org/ns/oa#" xmlns:dqv="http://www.w3.org/ns/dqv#"><edm:ProvidedCHO rdf:about="http://data.europeana.eu/item/73/S_OM_photo_OME004803"/><edm:WebRe

In [8]:
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import random

# Function to process a single subdirectory
def process_single_subdirectory(subdirectory_path):
    try:
        subdirectory = os.path.basename(subdirectory_path)
        print(f"Processing {subdirectory}...")

        # Get all files in the subdirectory
        files = os.listdir(subdirectory_path)
        size = len(files)
        print(f"Size of {subdirectory}: {size}")

        # Process RDF files in the sampled list
        output = fe.parse_rdf_files(subdirectory_path, subdirectory)

        # Write the output to an XML file
        output_file = f'/home/sbasir/Thesis/Thesis/EDP/collected_data/{subdirectory}'
        fe.write_data(output, output_file)

        print(f"Finished processing {subdirectory}. Output written to {output_file}")

    except Exception as e:
        print(f"Error processing {subdirectory}: {e}")

# Threaded function to process all subdirectories in the given directory
def process_zip_threaded(directory):
    subdirectories = [os.path.join(directory, subdir) for subdir in os.listdir(directory) if os.path.isdir(os.path.join(directory, subdir))]

    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {executor.submit(process_single_subdirectory, subdir): subdir for subdir in subdirectories}

        # Use tqdm to show progress as futures are completed
        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing subdirectories"):
            try:
                future.result()
            except Exception as e:
                subdirectory = futures[future]
                print(f"Error processing subdirectory {subdirectory}: {e}")

In [9]:
# Example usage
process_zip_threaded('/home/sbasir/Thesis/Thesis/EDP/selected_data')

Processing 73...
Processing 2021672...
Processing 728...


Processing subdirectories:   0%|          | 0/3 [00:00<?, ?it/s]

Size of 728: 816
Size of 2021672: 832
is translated
content is fine
passed checks
is translated
content is fine
passed checks
Size of 73: 3186
is translated
content is fine
passed checks


/home/sbasir/Thesis/Thesis/EDP/field_extractor.py:35: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on the current behaviour, change it to './/ore:proxyIn'
  element = tree.find(xpath_query, namespaces)
/home/sbasir/Thesis/Thesis/EDP/field_extractor.py:35: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on the current behaviour, change it to './/dcterms:modified'
  element = tree.find(xpath_query, namespaces)
/home/sbasir/Thesis/Thesis/EDP/field_extractor.py:35: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on the current behaviour, change it to './/edm:type'
  element = tree.find(xpath_query, namespaces)
/home/sbasir/Thesis/Thesis/EDP/field_extractor.py:35: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on th

is translated
content is fine
passed checks
is translated
content is fine
passed checks
does not exist
Failed to parse 864.xml: cannot unpack non-iterable bool object
does not exist
Failed to parse 405.xml: cannot unpack non-iterable bool object
does not exist
Failed to parse 42.xml: cannot unpack non-iterable bool object
does not exist
Failed to parse 2854.xml: cannot unpack non-iterable bool object
does not exist
Failed to parse 1502.xml: cannot unpack non-iterable bool object
does not exist
Failed to parse 1027.xml: cannot unpack non-iterable bool object
does not exist
Failed to parse 1663.xml: cannot unpack non-iterable bool object
does not exist
Failed to parse 1969.xml: cannot unpack non-iterable bool object
does not exist
Failed to parse 1394.xml: cannot unpack non-iterable bool object
does not exist
Failed to parse 762.xml: cannot unpack non-iterable bool object
does not exist
Failed to parse 2751.xml: cannot unpack non-iterable bool object
does not exist
Failed to parse 611.xm

...

is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks


/home/sbasir/Thesis/Thesis/EDP/field_extractor.py:35: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on the current behaviour, change it to './/ore:proxyIn'
  element = tree.find(xpath_query, namespaces)
/home/sbasir/Thesis/Thesis/EDP/field_extractor.py:35: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on the current behaviour, change it to './/dcterms:modified'
  element = tree.find(xpath_query, namespaces)
/home/sbasir/Thesis/Thesis/EDP/field_extractor.py:35: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on the current behaviour, change it to './/edm:type'
  element = tree.find(xpath_query, namespaces)
/home/sbasir/Thesis/Thesis/EDP/field_extractor.py:35: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on th

is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks


/home/sbasir/Thesis/Thesis/EDP/field_extractor.py:35: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on the current behaviour, change it to './/ore:proxyIn'
  element = tree.find(xpath_query, namespaces)
/home/sbasir/Thesis/Thesis/EDP/field_extractor.py:35: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on the current behaviour, change it to './/dcterms:modified'
  element = tree.find(xpath_query, namespaces)
/home/sbasir/Thesis/Thesis/EDP/field_extractor.py:35: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on the current behaviour, change it to './/edm:type'
  element = tree.find(xpath_query, namespaces)
/home/sbasir/Thesis/Thesis/EDP/field_extractor.py:35: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on th

is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
Finished processing 2021672. Output written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2021672
is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
is translated
content is fine
passed checks
Finished processing 728. Output written to /home/sbasir/Thesis/Thesis/EDP/collected_data/728


In [10]:
# def process_zip(directory):
#     # for each subdirectory in the UNZIP_DIR print the size of the subdirectory
#     for subdirectory in os.listdir(directory):
#         subdirectory_path = os.path.join(directory, subdirectory)
#         print(f"Size of {subdirectory}: {len(os.listdir(subdirectory_path))}")
#         print(subdirectory_path)
#         output = fe.parse_rdf_files(subdirectory_path)
#         fe.write_data(output, f'/home/sbasir/Thesis/Thesis/EDP/collected_data/{subdirectory}.xml')

In [11]:
# process_zip('/home/sbasir/Thesis/Thesis/EDP/selected_data')

In [12]:
# # Usage
# directory = '/home/sbasir/Thesis/Thesis/EDP/selected_data/254'  # Change this to your directory containing RDF/XML files
# solr_xml_data = fe.parse_rdf_files(directory)
# # solr_xml_data = fe.remove_duplicates(solr_xml_data)
# output_file = '/home/sbasir/Thesis/Thesis/EDP/testing.xml'
# fe.write_data(solr_xml_data, output_file)